In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()
df_ = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()

df = df.join(df_.set_index('did'), on='did')

dids = df['did'].to_numpy()
dates = np.array([datetime.fromisoformat(date) for date in df['date']])
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/cross_calibration/fdt/hmi.npz')
fdt, hmi = s['fdt'], s['hmi']

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [4]:
t = np.where(car_rot == 2298)[0]

files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))
files = files[t]
print(len(files))
print(files[0], files[-1])

269
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250516T122003_V202602220258_0545160505.fits.gz /home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250702T182009_V202602220334_0547020507.fits.gz


In [5]:
files_ = sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*'))

files = []
for file in files_:
    if fnmatch.fnmatch(file, '*blos_202505*'):
        files.append(file)

print(len(files))
print(files[0], files[-1])

228
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250501T002002_V202602220258_0545010501.fits.gz /home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250531T182003_V202602220334_0545310507.fits.gz


In [ ]:
for i in range(0, 200 + 1):

    nx, ny = 1024, 1024
    xc, yc = 511.5, 511.5
    Rsun = 511.5

    q = np.sin(i / 100 * np.pi) * 44.9

    #view_north = View(nx, ny, xc, yc, Rsun, -90, 90, 0, rsun_arc=0) ### North pole view
    view_north = View(nx, ny, xc, yc, Rsun, 0, 45, 0, rsun_arc=q * 3600) ### North pole view
    grid = view_north.grid()

    mean, variance, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

    plt.ioff()

    for file in files[:]:
        #print(file)

        with fits.open(file) as hdul:
            header = hdul[0].header.copy()
            data = hdul[0].data.copy()

        did = int(file.split('_')[-1].split('.')[0])
        data = undistort(data, header, xd, yd)

        view = View.from_header(header)
        view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0])

        transform = (view_north.to_carrington(correct_mu=True) -
                     view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=0.1, stonyhurst=False))
        grid_, alpha = transform(grid)
        data = bilinear(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4
        coverage += np.nan_to_num(n)
        mean_ = mean.copy()
        mean += np.nan_to_num((data - mean) * n / coverage)
        variance += np.nan_to_num((data - mean) * (data - mean_) * n)


    plt.ion()

    variance = variance / coverage
    mean[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    variance[coverage == 0] = np.nan

    show_data(mean, view_north, label=r'$B_{los}$', vmin=-40, vmax=40, to_file=f'temp1/{i:03d}.png')